In [ ]:
# Environment
import os, sys
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working: {os.getcwd()}")

# 08 - Census Data Extraction

**Goal**: Extract county + state demographics from US Census API.
**Source**: ACS 5-year estimates (2023)
**Output**: `census_counties.csv` + `census_demographics.csv`

In [ ]:
import pandas as pd
import requests
import os
from pathlib import Path

# Load API key
env_file = Path(".env")
if env_file.exists():
    for line in env_file.read_text().strip().split("\n"):
        if line.startswith("CENSUS_API_KEY="):
            os.environ["CENSUS_API_KEY"] = line.split("=", 1)[1]
CENSUS_KEY = os.environ.get("CENSUS_API_KEY", "")
print(f"API Key loaded: {bool(CENSUS_KEY)}")

## County-Level Data

Extract for all 28 counties matching promise locations.

In [ ]:
# Location → county FIPS mapping
CITY_TO_FIPS = {
    'Phoenix, AZ': ('04','013'), 'San Antonio, TX': ('48','029'), 'West Des Moines, IA': ('19','153'),
    'Mount Pleasant, WI': ('55','101'), 'Northern VA': ('51','059'), 'Portland, OR': ('41','051'),
    'Houston, TX': ('48','201'), 'Alto, TX': ('48','449'), 'DeSoto County, FL': ('12','031'),
    'Tompkins County, NY': ('36','109'), 'Columbus, OH': ('39','049'), 'Los Angeles, CA': ('06','037'),
    'Chicago, IL': ('17','031'), 'Secaucus, NJ': ('34','017'), 'Dallas, TX': ('48','113'),
    'Santa Clara, CA': ('06','085'), 'Ashburn, VA': ('51','107'), 'Denver, CO': ('08','031'),
    'Seattle, WA': ('53','033'), 'Inland Empire, CA': ('06','071'), 'New York, NY': ('36','061'),
    'San Diego, CA': ('06','073'), 'Austin, TX': ('48','453'), 'San Jose, CA': ('06','085'),
    'Riverside, CA': ('06','065'), 'Atlanta, GA': ('13','121'), 'Monmouth County, NJ': ('34','025'),
    'Las Vegas, NV': ('32','003'),
}

# ACS 5-year variables
VARS = [
    'NAME', 'B01003_001E',  # Total pop
    'B19013_001E',          # Median HH income
    'B25001_001E',          # Housing units
    'B25077_001E',          # Median home value
    'B25064_001E',          # Median rent
    'B23025_005E',          # Unemployed
    'B08006_001E',          # Workers 16+
    'B15003_022E',          # Bachelor's degree
    'B15003_025E',          # Doctorate
]

BASE = 'https://api.census.gov/data/2023/acs/acs5'

results = []
for loc, (st, ct) in CITY_TO_FIPS.items():
    params = {'get': ','.join(VARS), 'for': f'county:{ct}', 'in': f'state:{st}', 'key': CENSUS_KEY}
    r = requests.get(BASE, params=params, timeout=15)
    if r.ok and len(r.text) > 10:
        data = r.json()
        if len(data) > 1:
            row = data[1]
            results.append({
                'location': loc, 'county_name': row[0],
                'total_pop': int(row[1]), 'median_income': int(row[2]),
                'housing_units': int(row[3]), 'median_home_value': int(row[4]),
                'median_rent': int(row[5]), 'unemployed': int(row[6]),
                'workers_16plus': int(row[7]), 'bachelors_degree': int(row[8]),
                'doctorate': int(row[9]),
            })

df_counties = pd.DataFrame(results)
df_counties['unemployment_rate'] = round(df_counties['unemployed'] / df_counties['workers_16plus'] * 100, 2)
df_counties['pct_bachelors'] = round(df_counties['bachelors_degree'] / df_counties['workers_16plus'] * 100, 2)
df_counties.to_csv('data/raw/census_counties.csv', index=False)
print(f"✅ Counties: {len(df_counties)} × {len(df_counties.columns)}")

## State-Level Data

In [ ]:
# All states
params = {'get': ','.join(VARS), 'for': 'state:*', 'key': CENSUS_KEY}
r = requests.get(BASE, params=params, timeout=30)
data = r.json()

results = []
for row in data[1:]:
    results.append({
        'state_name': row[0], 'total_pop': int(row[1]), 'median_income': int(row[2]),
        'housing_units': int(row[3]), 'median_home_value': int(row[4]),
        'median_rent': int(row[5]), 'unemployed': int(row[6]),
        'workers_16plus': int(row[7]), 'bachelors_degree': int(row[8]), 'doctorate': int(row[9]),
    })

df_states = pd.DataFrame(results)
df_states['unemployment_rate'] = round(df_states['unemployed'] / df_states['workers_16plus'] * 100, 2)
df_states['pct_bachelors'] = round(df_states['bachelors_degree'] / df_states['workers_16plus'] * 100, 2)

# Add abbreviations
abbr = {'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','District of Columbia':'DC','Florida':'FL','Georgia':'GA','Hawaii':'HI',
    'Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH','New Jersey':'NJ',
    'New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK',
    'Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD',
    'Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA',
    'West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY','Puerto Rico':'PR'}
df_states['state_abbr'] = df_states['state_name'].map(abbr)
df_states = df_states[['state_abbr','state_name','total_pop','median_income','housing_units',
                        'median_home_value','median_rent','unemployed','workers_16plus',
                        'bachelors_degree','doctorate','unemployment_rate','pct_bachelors']]

df_states.to_csv('data/raw/census_demographics.csv', index=False)
print(f"✅ States: {len(df_states)} × {len(df_states.columns)}")

## Summary
✅ Extracted: counties (28) + states (52), 13 variables each